In [2]:
# ---------------------------------------------------------------------
# CELL 1: SETUP
# rsds → PET pipeline initialization
# ---------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# ROOT PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RASTERS_DIR = PROJECT_ROOT / "rasters"

CMIP6_SCENARIOS_DIR = DATA_DIR / "scenarios"
ERA5_DIR = DATA_DIR / "simulated" / "era5_land"

# Reference variable
ERA5_PET_DIR = ERA5_DIR / "pet"

# Outputs
CORRECTED_ROOT = RASTERS_DIR / "corrected" / "cmip6"

ARTIFACTS_DIR = DATA_DIR / "artifacts"
DATASETS_DIR = ARTIFACTS_DIR / "datasets"
MODELS_DIR = ARTIFACTS_DIR / "models"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
SCENARIOS = ["ssp245", "ssp370", "ssp585"]

HIST_START = 1981
HIST_END = 2014

FUTURE_START = 2026
FUTURE_END = 2050

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"rsds_pet_{RUN_TS}"

# ---------------------------------------------------------------------
# ENSURE OUTPUT FOLDERS EXIST
# ---------------------------------------------------------------------
for scenario in SCENARIOS:
    (CORRECTED_ROOT / scenario / "pet").mkdir(parents=True, exist_ok=True)

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def yyyymm_from_name(path: Path):
    stem = path.stem
    parts = stem.split("_")

    for part in reversed(parts):
        if len(part) == 6 and part.isdigit():
            return part

    return None


def list_tifs(folder: Path):
    if not folder.exists():
        return []
    return sorted(folder.glob("*.tif"))


def index_by_yyyymm(folder: Path):
    out = {}

    for fp in list_tifs(folder):
        key = yyyymm_from_name(fp)
        if key:
            out[key] = fp

    return out


def filter_period(index: dict, start_year: int, end_year: int):
    return {
        ym: fp
        for ym, fp in index.items()
        if start_year <= int(ym[:4]) <= end_year
    }


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


# ---------------------------------------------------------------------
# LOAD ERA5 PET INDEX
# ---------------------------------------------------------------------
era5_pet_index = index_by_yyyymm(ERA5_PET_DIR)

era5_pet_hist = filter_period(
    era5_pet_index,
    HIST_START,
    HIST_END
)

# ---------------------------------------------------------------------
# STATUS PRINT
# ---------------------------------------------------------------------
print("=" * 60)
print("RSDS → PET PIPELINE SETUP")
print("=" * 60)

print(f"PROJECT ROOT : {PROJECT_ROOT}")
print(f"ERA5 PET DIR : {ERA5_PET_DIR}")
print(f"SCENARIOS    : {SCENARIOS}")

print("\nERA5 PET STATUS")
print(f"Total months     : {len(era5_pet_index)}")
print(f"Historical months: {len(era5_pet_hist)}")

if era5_pet_hist:
    print(f"First hist month : {min(era5_pet_hist.keys())}")
    print(f"Last hist month  : {max(era5_pet_hist.keys())}")

print("\nRUN INFO")
print(f"Run ID   : {RUN_ID}")
print(f"Timestamp: {RUN_TS}")

RSDS → PET PIPELINE SETUP
PROJECT ROOT : C:\Projects\Infer RozviDrought
ERA5 PET DIR : C:\Projects\Infer RozviDrought\data\simulated\era5_land\pet
SCENARIOS    : ['ssp245', 'ssp370', 'ssp585']

ERA5 PET STATUS
Total months     : 523
Historical months: 400
First hist month : 198109
Last hist month  : 201412

RUN INFO
Run ID   : rsds_pet_20260322T111644Z
Timestamp: 20260322T111644Z


In [3]:
# ---------------------------------------------------------------------
# CELL 2: INSPECT RSDS NETCDF PROPERTIES
# This cell only confirms rsds properties before any PET derivation.
# ---------------------------------------------------------------------

print("=" * 60)
print("RSDS NETCDF PROPERTY INSPECTION")
print("=" * 60)

scenario_rsds_nc = {}

for scenario in SCENARIOS:

    print("\n" + "-" * 60)
    print(f"SCENARIO: {scenario}")
    print("-" * 60)

    rsds_dir = CMIP6_SCENARIOS_DIR / scenario / "rsds"

    nc_files = sorted(
        f for f in rsds_dir.glob("*.nc")
        if "Amon" in f.name
    )

    if not nc_files:
        print("No valid RSDS NetCDF file found.")
        continue

    nc_path = nc_files[0]
    scenario_rsds_nc[scenario] = nc_path

    print(f"File: {nc_path}")

    ds = xr.open_dataset(nc_path, engine="netcdf4")

    print("\nDataset summary:")
    print(ds)

    print("\nVariables:")
    for name, da in ds.data_vars.items():
        print(
            f"- {name}: dims={da.dims}, shape={da.shape}, units={da.attrs.get('units')}"
        )

    print("\nCoordinates:")
    for name, da in ds.coords.items():
        print(
            f"- {name}: dims={da.dims}, shape={da.shape}"
        )

    if "time" in ds.coords:
        time_vals = pd.to_datetime(ds["time"].values)
        print("\nTIME COVERAGE")
        print(f"Start : {time_vals[0]}")
        print(f"End   : {time_vals[-1]}")
        print(f"Months: {len(time_vals)}")

    if "rsds" in ds.data_vars:
        arr0 = ds["rsds"].isel(time=0).values
        print("\nFIRST TIMESTEP STATS")
        print(f"Min  : {np.nanmin(arr0)}")
        print(f"Max  : {np.nanmax(arr0)}")
        print(f"Mean : {np.nanmean(arr0)}")

    ds.close()

print("\n" + "=" * 60)
print("RSDS INSPECTION COMPLETE")
print("=" * 60)

RSDS NETCDF PROPERTY INSPECTION

------------------------------------------------------------
SCENARIO: ssp245
------------------------------------------------------------
File: C:\Projects\Infer RozviDrought\data\scenarios\ssp245\rsds\rsds_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc

Dataset summary:
<xarray.Dataset> Size: 24MB
Dimensions:    (time: 300, bnds: 2, lat: 96, lon: 192)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2026-01-16T12:00:00 ... 2050-12-16T1...
  * lat        (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
  * lon        (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 354.4 356.2 358.1
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 5kB ...
    lat_bnds   (time, lat, bnds) float64 461kB ...
    lon_bnds   (time, lon, bnds) float64 922kB ...
    rsds       (time, lat, lon) float32 22MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            Scena

In [4]:
# ---------------------------------------------------------------------
# CELL 3: INSPECT ERA5 PET REFERENCE PROPERTIES
# This cell only confirms PET reference properties.
# ---------------------------------------------------------------------

print("=" * 60)
print("ERA5 PET REFERENCE INSPECTION")
print("=" * 60)

pet_months = sorted(era5_pet_hist.keys())

print(f"Historical PET months: {len(pet_months)}")
print(f"First month          : {pet_months[0]}")
print(f"Last month           : {pet_months[-1]}")

sample_month = pet_months[0]
sample_path = era5_pet_hist[sample_month]

print("\nSample file:")
print(sample_path)

with rasterio.open(sample_path) as src:
    arr = src.read(1).astype(np.float32)

    print("\nRASTER PROPERTIES")
    print(f"CRS       : {src.crs}")
    print(f"Shape     : ({src.height}, {src.width})")
    print(f"Count     : {src.count}")
    print(f"Dtype     : {src.dtypes[0]}")
    print(f"Nodata    : {src.nodata}")
    print(f"Bounds    : {src.bounds}")
    print(f"Transform : {src.transform}")

    print("\nVALUE STATS")
    print(f"Min       : {np.nanmin(arr)}")
    print(f"Max       : {np.nanmax(arr)}")
    print(f"Mean      : {np.nanmean(arr)}")

print("\n" + "=" * 60)
print("ERA5 PET INSPECTION COMPLETE")
print("=" * 60)

ERA5 PET REFERENCE INSPECTION
Historical PET months: 400
First month          : 198109
Last month           : 201412

Sample file:
C:\Projects\Infer RozviDrought\data\simulated\era5_land\pet\pet_198109.tif

RASTER PROPERTIES
CRS       : EPSG:4326
Shape     : (70, 80)
Count     : 1
Dtype     : float32
Nodata    : None
Bounds    : BoundingBox(left=25.100114814474466, bottom=-22.500102921341654, right=33.100151408729275, top=-15.500070901368694)
Transform : | 0.10, 0.00, 25.10|
| 0.00,-0.10,-15.50|
| 0.00, 0.00, 1.00|

VALUE STATS
Min       : -0.0020295504946261644
Max       : -0.00011624846956692636
Mean      : -0.0004149726592004299

ERA5 PET INSPECTION COMPLETE


In [5]:
# ---------------------------------------------------------------------
# CELL 4: CHECK PET SIGN CONVENTION AND BUILD NORMALIZED REFERENCE VIEW
# ERA5 evap/pet can be stored as negative upward flux.
# This cell confirms that, then builds a positive PET view for modelling.
# ---------------------------------------------------------------------

print("=" * 60)
print("ERA5 PET SIGN CHECK + NORMALIZATION")
print("=" * 60)

check_months = []
all_hist_months = sorted(era5_pet_hist.keys())

# sample a few months across the record
if len(all_hist_months) >= 4:
    check_months = [
        all_hist_months[0],
        all_hist_months[len(all_hist_months) // 3],
        all_hist_months[(2 * len(all_hist_months)) // 3],
        all_hist_months[-1],
    ]
else:
    check_months = all_hist_months

summary_rows = []

for month in check_months:
    fp = era5_pet_hist[month]

    with rasterio.open(fp) as src:
        arr = src.read(1).astype(np.float32)

    raw_min = float(np.nanmin(arr))
    raw_max = float(np.nanmax(arr))
    raw_mean = float(np.nanmean(arr))

    # normalize sign: PET magnitude should be positive for our workflow
    pet_pos = np.where(np.isfinite(arr), -1.0 * arr, np.nan)

    pos_min = float(np.nanmin(pet_pos))
    pos_max = float(np.nanmax(pet_pos))
    pos_mean = float(np.nanmean(pet_pos))

    summary_rows.append({
        "yyyymm": month,
        "raw_min": raw_min,
        "raw_max": raw_max,
        "raw_mean": raw_mean,
        "pos_min": pos_min,
        "pos_max": pos_max,
        "pos_mean": pos_mean,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)

# simple decision rule
all_negative_mean = bool((summary_df["raw_mean"] < 0).all())
all_positive_norm = bool((summary_df["pos_mean"] > 0).all())

print("\n" + "=" * 60)
print("DECISION")
print("=" * 60)
print(f"All sampled raw means negative     : {all_negative_mean}")
print(f"All normalized means positive      : {all_positive_norm}")

PET_REFERENCE_SIGN_FLIP = True

print(f"\nPET_REFERENCE_SIGN_FLIP = {PET_REFERENCE_SIGN_FLIP}")
print("Interpretation: use pet_ref = -1 * raw_pet for comparison/training.")

ERA5 PET SIGN CHECK + NORMALIZATION
   yyyymm   raw_min   raw_max  raw_mean   pos_min   pos_max  pos_mean
0  198109 -0.002030 -0.000116 -0.000415  0.000116  0.002030  0.000415
1  199210 -0.002214 -0.000191 -0.000535  0.000191  0.002214  0.000535
2  200311 -0.001974 -0.000218 -0.000480  0.000218  0.001974  0.000480
3  201412 -0.001073 -0.000185 -0.000316  0.000185  0.001073  0.000316

DECISION
All sampled raw means negative     : True
All normalized means positive      : True

PET_REFERENCE_SIGN_FLIP = True
Interpretation: use pet_ref = -1 * raw_pet for comparison/training.


In [6]:
# ---------------------------------------------------------------------
# CELL 5: INSPECT HISTORICAL CMIP6 RSDS FOR TRAINING PERIOD
# This confirms the historical rsds source before PET derivation/training.
# ---------------------------------------------------------------------

print("=" * 60)
print("HISTORICAL CMIP6 RSDS INSPECTION")
print("=" * 60)

HIST_CMIP6_RSDS_DIR = DATA_DIR / "historical" / "cmip6" / "rsds"

hist_nc_files = sorted(
    f for f in HIST_CMIP6_RSDS_DIR.glob("*.nc")
    if "Amon" in f.name
)

print(f"Directory   : {HIST_CMIP6_RSDS_DIR}")
print(f"NC files    : {len(hist_nc_files)}")

if not hist_nc_files:
    raise FileNotFoundError("No historical CMIP6 rsds NetCDF file found.")

hist_rsds_nc = hist_nc_files[0]
print(f"Using file  : {hist_rsds_nc}")

ds = xr.open_dataset(hist_rsds_nc, engine="netcdf4")

print("\nDataset summary:")
print(ds)

print("\nVariables:")
for name, da in ds.data_vars.items():
    print(f"- {name}: dims={da.dims}, shape={da.shape}, units={da.attrs.get('units')}")

time_vals = pd.to_datetime(ds["time"].values)
print("\nTIME COVERAGE")
print(f"Start : {time_vals[0]}")
print(f"End   : {time_vals[-1]}")
print(f"Months: {len(time_vals)}")

arr0 = ds["rsds"].isel(time=0).values
print("\nFIRST TIMESTEP STATS")
print(f"Min  : {np.nanmin(arr0)}")
print(f"Max  : {np.nanmax(arr0)}")
print(f"Mean : {np.nanmean(arr0)}")

ds.close()

print("\n" + "=" * 60)
print("HISTORICAL RSDS INSPECTION COMPLETE")
print("=" * 60)

HISTORICAL CMIP6 RSDS INSPECTION
Directory   : C:\Projects\Infer RozviDrought\data\historical\cmip6\rsds
NC files    : 1
Using file  : C:\Projects\Infer RozviDrought\data\historical\cmip6\rsds\rsds_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19810116-20141216.nc

Dataset summary:
<xarray.Dataset> Size: 32MB
Dimensions:    (time: 408, bnds: 2, lat: 96, lon: 192)
Coordinates:
  * time       (time) datetime64[ns] 3kB 1981-01-16T12:00:00 ... 2014-12-16T1...
  * lat        (lat) float64 768B -88.57 -86.72 -84.86 ... 84.86 86.72 88.57
  * lon        (lon) float64 2kB 0.0 1.875 3.75 5.625 ... 354.4 356.2 358.1
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 7kB ...
    lat_bnds   (time, lat, bnds) float64 627kB ...
    lon_bnds   (time, lon, bnds) float64 1MB ...
    rsds       (time, lat, lon) float32 30MB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    

In [7]:
# ---------------------------------------------------------------------
# CELL 6: SAME-MONTH GRID ALIGNMENT CHECK
# Reproject one historical CMIP6 rsds month to the ERA5 PET grid.
# This is only a spatial/time alignment check, not PET derivation yet.
# ---------------------------------------------------------------------

from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

print("=" * 60)
print("SAME-MONTH RSDS ↔ PET ALIGNMENT CHECK")
print("=" * 60)

# pick first overlapping month from ERA5 historical PET
sample_month = sorted(era5_pet_hist.keys())[0]
sample_year = int(sample_month[:4])
sample_mon = int(sample_month[4:])

print(f"Sample month: {sample_month}")

# ERA5 PET reference
pet_ref_path = era5_pet_hist[sample_month]

with rasterio.open(pet_ref_path) as ref:
    pet_raw = ref.read(1).astype(np.float32)
    pet_ref = np.where(np.isfinite(pet_raw), -1.0 * pet_raw, np.nan)  # normalized positive PET
    dst_shape = (ref.height, ref.width)
    dst_transform = ref.transform
    dst_crs = ref.crs

print(f"ERA5 PET shape : {dst_shape}")
print(f"ERA5 PET mean  : {np.nanmean(pet_ref)}")

# historical CMIP6 rsds
ds = xr.open_dataset(hist_rsds_nc, engine="netcdf4")
time_vals = pd.to_datetime(ds["time"].values)

match_idx = np.where(
    (time_vals.year == sample_year) &
    (time_vals.month == sample_mon)
)[0]

if len(match_idx) == 0:
    ds.close()
    raise ValueError(f"No matching historical rsds month found for {sample_month}")

idx = int(match_idx[0])

rsds = ds["rsds"].isel(time=idx).values.astype(np.float32)
lat = ds["lat"].values
lon = ds["lon"].values
ds.close()

# convert lon 0..360 to -180..180 and sort
lon_180 = np.where(lon > 180.0, lon - 360.0, lon)
sort_idx = np.argsort(lon_180)
lon_180 = lon_180[sort_idx]
rsds = rsds[:, sort_idx]

# flip latitude to descending order for raster convention
if lat[0] < lat[-1]:
    lat = lat[::-1]
    rsds = rsds[::-1, :]

# clean tiny negatives
rsds = np.where(rsds < 0, 0.0, rsds)

# source transform
lon_res = float(np.abs(np.diff(lon_180).mean()))
lat_res = float(np.abs(np.diff(lat).mean()))

src_transform = from_bounds(
    lon_180.min() - lon_res / 2.0,
    lat.min() - lat_res / 2.0,
    lon_180.max() + lon_res / 2.0,
    lat.max() + lat_res / 2.0,
    rsds.shape[1],
    rsds.shape[0],
)

# reproject rsds to ERA5 PET grid
rsds_regridded = np.full(dst_shape, np.nan, dtype=np.float32)

reproject(
    source=rsds,
    destination=rsds_regridded,
    src_transform=src_transform,
    src_crs="EPSG:4326",
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.bilinear,
    src_nodata=None,
    dst_nodata=np.nan,
)

print("\nRESULT")
print(f"Regridded rsds shape : {rsds_regridded.shape}")
print(f"Regridded rsds min   : {np.nanmin(rsds_regridded)}")
print(f"Regridded rsds max   : {np.nanmax(rsds_regridded)}")
print(f"Regridded rsds mean  : {np.nanmean(rsds_regridded)}")

print("\nConclusion: same-month, same-grid alignment is ready for PET derivation.")

SAME-MONTH RSDS ↔ PET ALIGNMENT CHECK
Sample month: 198109
ERA5 PET shape : (70, 80)
ERA5 PET mean  : 0.0004149726592004299

RESULT
Regridded rsds shape : (70, 80)
Regridded rsds min   : 253.6083221435547
Regridded rsds max   : 287.25347900390625
Regridded rsds mean  : 276.43865966796875

Conclusion: same-month, same-grid alignment is ready for PET derivation.


In [8]:
# ---------------------------------------------------------------------
# CELL 7: PET DERIVATION READINESS CHECK
# We confirm what extra drivers are available, because rsds alone is not enough.
# ---------------------------------------------------------------------

print("=" * 60)
print("PET DERIVATION READINESS CHECK")
print("=" * 60)

required_future = {
    "tas": DATA_DIR / "scenarios" / "ssp245" / "tas",
    "rsds": DATA_DIR / "scenarios" / "ssp245" / "rsds",
    "huss": DATA_DIR / "scenarios" / "ssp245" / "huss",
}

required_ref = {
    "pet_ref": DATA_DIR / "simulated" / "era5_land" / "pet",
    "t2m_ref": DATA_DIR / "simulated" / "era5_land" / "t2m",
    "d2m_ref": DATA_DIR / "simulated" / "era5_land" / "d2m",
}

print("CMIP6 future inputs")
for name, path in required_future.items():
    print(f"{name:10s} | exists={path.exists()} | path={path}")

print("\nERA5 reference inputs")
for name, path in required_ref.items():
    tif_count = len(list(path.glob('*.tif'))) if path.exists() else 0
    print(f"{name:10s} | exists={path.exists()} | tif_count={tif_count} | path={path}")

print("\nDecision:")
print("Use a temperature-radiation-humidity based PET approximation, then bias-correct to ERA5 PET.")

PET DERIVATION READINESS CHECK
CMIP6 future inputs
tas        | exists=True | path=C:\Projects\Infer RozviDrought\data\scenarios\ssp245\tas
rsds       | exists=True | path=C:\Projects\Infer RozviDrought\data\scenarios\ssp245\rsds
huss       | exists=True | path=C:\Projects\Infer RozviDrought\data\scenarios\ssp245\huss

ERA5 reference inputs
pet_ref    | exists=True | tif_count=523 | path=C:\Projects\Infer RozviDrought\data\simulated\era5_land\pet
t2m_ref    | exists=True | tif_count=523 | path=C:\Projects\Infer RozviDrought\data\simulated\era5_land\t2m
d2m_ref    | exists=True | tif_count=523 | path=C:\Projects\Infer RozviDrought\data\simulated\era5_land\d2m

Decision:
Use a temperature-radiation-humidity based PET approximation, then bias-correct to ERA5 PET.


In [9]:
# ---------------------------------------------------------------------
# CELL 8: BUILD A FIRST HISTORICAL PET PROXY (SAME MONTH, SAME GRID)
# This is a practical proxy, not the final corrected PET.
# We use temperature + radiation + humidity signal, then compare to ERA5 PET.
# ---------------------------------------------------------------------

from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

print("=" * 60)
print("HISTORICAL PET PROXY CHECK")
print("=" * 60)

# ------------------------------------------------------------
# Historical source files
# ------------------------------------------------------------
HIST_CMIP6_TAS_DIR = DATA_DIR / "historical" / "cmip6" / "tas"
HIST_CMIP6_HUSS_DIR = DATA_DIR / "historical" / "cmip6" / "huss"

hist_tas_nc = sorted(f for f in HIST_CMIP6_TAS_DIR.glob("*.nc") if "Amon" in f.name)[0]
hist_huss_nc = sorted(f for f in HIST_CMIP6_HUSS_DIR.glob("*.nc") if "Amon" in f.name)[0]

sample_month = sorted(era5_pet_hist.keys())[0]
sample_year = int(sample_month[:4])
sample_mon = int(sample_month[4:])

print(f"Sample month : {sample_month}")

# ------------------------------------------------------------
# ERA5 PET reference (normalized positive)
# ------------------------------------------------------------
pet_ref_path = era5_pet_hist[sample_month]

with rasterio.open(pet_ref_path) as ref:
    pet_raw = ref.read(1).astype(np.float32)
    pet_ref = np.where(np.isfinite(pet_raw), -1.0 * pet_raw, np.nan)
    dst_shape = (ref.height, ref.width)
    dst_transform = ref.transform
    dst_crs = ref.crs

print(f"ERA5 PET mean : {np.nanmean(pet_ref)}")

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------
def get_month_slice(nc_path, var_name, year, month):
    ds = xr.open_dataset(nc_path, engine="netcdf4")
    time_vals = pd.to_datetime(ds["time"].values)
    idx = np.where((time_vals.year == year) & (time_vals.month == month))[0][0]
    arr = ds[var_name].isel(time=idx).values.astype(np.float32)
    lat = ds["lat"].values
    lon = ds["lon"].values
    ds.close()
    return arr, lat, lon

def prep_cmip6_grid(arr, lat, lon):
    lon_180 = np.where(lon > 180.0, lon - 360.0, lon)
    sort_idx = np.argsort(lon_180)
    lon_180 = lon_180[sort_idx]
    arr = arr[:, sort_idx]

    if lat[0] < lat[-1]:
        lat = lat[::-1]
        arr = arr[::-1, :]

    return arr, lat, lon_180

def build_src_transform(lat_desc, lon_180, height, width):
    lon_res = float(np.abs(np.diff(lon_180).mean()))
    lat_res = float(np.abs(np.diff(lat_desc).mean()))
    return from_bounds(
        lon_180.min() - lon_res / 2.0,
        lat_desc.min() - lat_res / 2.0,
        lon_180.max() + lon_res / 2.0,
        lat_desc.max() + lat_res / 2.0,
        width,
        height,
    )

def regrid_to_era5(arr, lat, lon):
    arr, lat, lon = prep_cmip6_grid(arr, lat, lon)
    src_transform = build_src_transform(lat, lon, arr.shape[0], arr.shape[1])
    out = np.full(dst_shape, np.nan, dtype=np.float32)

    reproject(
        source=arr,
        destination=out,
        src_transform=src_transform,
        src_crs="EPSG:4326",
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        dst_nodata=np.nan,
    )
    return out

# ------------------------------------------------------------
# Load historical CMIP6 drivers for same month
# ------------------------------------------------------------
tas_src, tas_lat, tas_lon = get_month_slice(hist_tas_nc, "tas", sample_year, sample_mon)
rsds_src, rsds_lat, rsds_lon = get_month_slice(hist_rsds_nc, "rsds", sample_year, sample_mon)
huss_src, huss_lat, huss_lon = get_month_slice(hist_huss_nc, "huss", sample_year, sample_mon)

# clean tiny negative rsds
rsds_src = np.where(rsds_src < 0, 0.0, rsds_src)

# ------------------------------------------------------------
# Regrid to ERA5 PET grid
# ------------------------------------------------------------
tas = regrid_to_era5(tas_src, tas_lat, tas_lon)      # Kelvin
rsds = regrid_to_era5(rsds_src, rsds_lat, rsds_lon)  # W/m2
huss = regrid_to_era5(huss_src, huss_lat, huss_lon)  # kg/kg

# ------------------------------------------------------------
# Build simple PET proxy
# Practical proxy:
#   more radiation -> more PET
#   warmer air     -> more PET
#   more humidity  -> slightly less PET
# Scale kept arbitrary here; bias correction will learn final mapping.
# ------------------------------------------------------------
tas_c = tas - 273.15
humidity_penalty = 1.0 - np.clip(huss / 0.03, 0.0, 0.95)

pet_proxy = np.where(
    np.isfinite(tas_c) & np.isfinite(rsds) & np.isfinite(huss),
    np.maximum(0.0, rsds) * np.maximum(0.0, tas_c + 5.0) * humidity_penalty,
    np.nan
).astype(np.float32)

print("\nPROXY STATS")
print(f"TAS mean (C)      : {np.nanmean(tas_c)}")
print(f"RSDS mean (W/m2)  : {np.nanmean(rsds)}")
print(f"HUSS mean         : {np.nanmean(huss)}")
print(f"PET proxy mean    : {np.nanmean(pet_proxy)}")
print(f"ERA5 PET mean     : {np.nanmean(pet_ref)}")

print("\nConclusion: proxy built successfully; next step is to build the full historical training table and let the bias model learn the mapping to ERA5 PET.")

HISTORICAL PET PROXY CHECK
Sample month : 198109
ERA5 PET mean : 0.0004149726592004299

PROXY STATS
TAS mean (C)      : 22.002010345458984
RSDS mean (W/m2)  : 276.43865966796875
HUSS mean         : 0.006165369413793087
PET proxy mean    : 5939.92236328125
ERA5 PET mean     : 0.0004149726592004299

Conclusion: proxy built successfully; next step is to build the full historical training table and let the bias model learn the mapping to ERA5 PET.


In [10]:
# ---------------------------------------------------------------------
# CELL 9: BUILD HISTORICAL PET TRAINING DATASET
# Creates a training table for learning PET from CMIP6 drivers/proxy
# against normalized ERA5 PET.
# ---------------------------------------------------------------------

from datetime import datetime, timezone

print("=" * 60)
print("BUILD HISTORICAL PET TRAINING DATASET")
print("=" * 60)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PET_DATASET_PATH = DATASETS_DIR / f"cmip6_pet_training_dataset_{timestamp}.parquet"

hist_months = sorted(era5_pet_hist.keys())
records = []

print(f"Months to process: {len(hist_months)}")

for month in hist_months:
    year = int(month[:4])
    mon = int(month[4:])

    # ERA5 PET reference (normalized positive)
    pet_ref_path = era5_pet_hist[month]
    with rasterio.open(pet_ref_path) as ref:
        pet_raw = ref.read(1).astype(np.float32)
        pet_ref = np.where(np.isfinite(pet_raw), -1.0 * pet_raw, np.nan)

    # CMIP6 monthly slices
    tas_src, tas_lat, tas_lon = get_month_slice(hist_tas_nc, "tas", year, mon)
    rsds_src, rsds_lat, rsds_lon = get_month_slice(hist_rsds_nc, "rsds", year, mon)
    huss_src, huss_lat, huss_lon = get_month_slice(hist_huss_nc, "huss", year, mon)

    rsds_src = np.where(rsds_src < 0, 0.0, rsds_src)

    # Regrid to ERA5 grid
    tas = regrid_to_era5(tas_src, tas_lat, tas_lon)      # K
    rsds = regrid_to_era5(rsds_src, rsds_lat, rsds_lon)  # W/m2
    huss = regrid_to_era5(huss_src, huss_lat, huss_lon)  # kg/kg

    # Proxy
    tas_c = tas - 273.15
    humidity_penalty = 1.0 - np.clip(huss / 0.03, 0.0, 0.95)

    pet_proxy = np.where(
        np.isfinite(tas_c) & np.isfinite(rsds) & np.isfinite(huss),
        np.maximum(0.0, rsds) * np.maximum(0.0, tas_c + 5.0) * humidity_penalty,
        np.nan
    ).astype(np.float32)

    # Flatten
    df = pd.DataFrame({
        "yyyymm": month,
        "pixel_id": np.arange(pet_ref.size, dtype=np.int32),
        "tas_k": tas.flatten(),
        "tas_c": tas_c.flatten(),
        "rsds_wm2": rsds.flatten(),
        "huss": huss.flatten(),
        "pet_proxy": pet_proxy.flatten(),
        "pet_era5": pet_ref.flatten(),
    })

    records.append(df)
    print(f"Processed: {month}")

pet_train_df = pd.concat(records, ignore_index=True)
pet_train_df = pet_train_df.replace([np.inf, -np.inf], np.nan)

PET_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
pet_train_df.to_parquet(PET_DATASET_PATH, index=False)

print("\n" + "=" * 60)
print("PET TRAINING DATASET SAVED")
print("=" * 60)
print(f"Rows : {len(pet_train_df)}")
print(f"Path : {PET_DATASET_PATH}")
print(f"Columns : {list(pet_train_df.columns)}")

BUILD HISTORICAL PET TRAINING DATASET
Months to process: 400
Processed: 198109
Processed: 198110
Processed: 198111
Processed: 198112
Processed: 198201
Processed: 198202
Processed: 198203
Processed: 198204
Processed: 198205
Processed: 198206
Processed: 198207
Processed: 198208
Processed: 198209
Processed: 198210
Processed: 198211
Processed: 198212
Processed: 198301
Processed: 198302
Processed: 198303
Processed: 198304
Processed: 198305
Processed: 198306
Processed: 198307
Processed: 198308
Processed: 198309
Processed: 198310
Processed: 198311
Processed: 198312
Processed: 198401
Processed: 198402
Processed: 198403
Processed: 198404
Processed: 198405
Processed: 198406
Processed: 198407
Processed: 198408
Processed: 198409
Processed: 198410
Processed: 198411
Processed: 198412
Processed: 198501
Processed: 198502
Processed: 198503
Processed: 198504
Processed: 198505
Processed: 198506
Processed: 198507
Processed: 198508
Processed: 198509
Processed: 198510
Processed: 198511
Processed: 198512
Pro

In [11]:
# ---------------------------------------------------------------------
# SAVE PET TRAINING DATASET MANIFEST
# This records provenance for the PET training dataset.
# ---------------------------------------------------------------------

from datetime import datetime, timezone
import json

print("=" * 60)
print("SAVE PET TRAINING DATASET MANIFEST")
print("=" * 60)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

DATASET_PATH = Path(
    r"C:\Projects\Infer RozviDrought\data\artifacts\datasets"
) / "cmip6_pet_training_dataset_20260322T113428Z.parquet"

MANIFEST_PATH = (
    MANIFESTS_DIR
    / f"cmip6_pet_training_dataset_{RUN_TS}_manifest.json"
)

manifest = {
    "dataset_name": DATASET_PATH.name,
    "dataset_path": str(DATASET_PATH),
    "created_utc": RUN_TS,
    "rows": 2240000,
    "columns": [
        "yyyymm",
        "pixel_id",
        "tas_k",
        "tas_c",
        "rsds_wm2",
        "huss",
        "pet_proxy",
        "pet_era5",
    ],
    "training_period": {
        "start": "198109",
        "end": "201412",
        "months": 400
    },
    "source_variables": [
        "CMIP6 tas",
        "CMIP6 rsds",
        "CMIP6 huss",
        "ERA5 pet (normalized positive)"
    ],
    "grid_reference": {
        "crs": "EPSG:4326",
        "shape": [70, 80]
    }
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 60)
print("MANIFEST SAVED")
print("=" * 60)
print(f"Path : {MANIFEST_PATH}")

SAVE PET TRAINING DATASET MANIFEST

MANIFEST SAVED
Path : C:\Projects\Infer RozviDrought\data\artifacts\manifests\cmip6_pet_training_dataset_20260322T114711Z_manifest.json


In [2]:
# ---------------------------------------------------------------------
# VALIDATION CELL
# This cell is intentionally self-contained.
# It is used only to validate corrected PET rasters after:
#   1) PET model training
#   2) future raster correction
# ---------------------------------------------------------------------

from pathlib import Path
import rasterio
import numpy as np

# ---------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

CORRECTED_ROOT = (
    PROJECT_ROOT
    / "rasters"
    / "corrected"
    / "cmip6"
)

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

print("=" * 60)
print("CMIP6 CORRECTED PET OUTPUT VALIDATION")
print("=" * 60)

results = []

for scenario in SCENARIOS:

    pet_dir = CORRECTED_ROOT / scenario / "pet"

    if not pet_dir.exists():
        print(f"{scenario}: directory missing")
        continue

    tif_files = sorted(pet_dir.glob("*.tif"))

    if not tif_files:
        print(f"{scenario}: no rasters found")
        continue

    sample_file = tif_files[0]

    with rasterio.open(sample_file) as src:
        arr = src.read(1).astype(np.float32)

        stats = {
            "scenario": scenario,
            "variable": "pet",
            "status": "ok",
            "months_found": len(tif_files),
            "sample_month": sample_file.stem.split("_")[-1],
            "sample_file": sample_file.name,
            "crs": str(src.crs),
            "shape": (src.height, src.width),
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
        }

        results.append(stats)
        print(stats)

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

CMIP6 CORRECTED PET OUTPUT VALIDATION
{'scenario': 'ssp245', 'variable': 'pet', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'pet_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 0.00026897049974650145, 'max': 0.0003817626857198775, 'mean': 0.0002981182187795639}
{'scenario': 'ssp370', 'variable': 'pet', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'pet_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 0.0002882957342080772, 'max': 0.0004307978961151093, 'mean': 0.0003558826574590057}
{'scenario': 'ssp585', 'variable': 'pet', 'status': 'ok', 'months_found': 300, 'sample_month': '202601', 'sample_file': 'pet_corrected_202601.tif', 'crs': 'EPSG:4326', 'shape': (70, 80), 'min': 0.00028072818531654775, 'max': 0.000408267107559368, 'mean': 0.0003407915064599365}

VALIDATION COMPLETE
